# Библиотеки

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import pingouin as pg
import seaborn as sns

## 9.17 Помиксуем-ка мы сами...: парные и независимые группы вместе

In [2]:
df = pg.read_dataset('mixed_anova')

In [4]:
df.sample(5)

,Scores,Time,Group,Subject
173,5.160928,June,Meditation,53
126,5.320676,January,Meditation,36
36,5.624713,January,Control,6
97,6.028288,August,Meditation,37
98,4.431011,August,Meditation,38


In [6]:
pg.mixed_anova(data=df, dv='Scores', between='Group', within='Time', subject='Subject', effsize='ng2')

,Source,SS,DF1,DF2,MS,F,p-unc,ng2,eps
0,Group,5.459963,1,58,5.459963,5.051709,0.028420,0.030673,NaN
1,Time,7.628428,2,116,3.814214,4.027394,0.020369,0.042339,0.998751
2,Interaction,5.167192,2,116,2.583596,2.727996,0.069545,0.029076,NaN


In [7]:
pg.homoscedasticity(df, dv='Scores', group='Group') # Проверяем равенство дисперсий

,W,pval,equal_var
levene,0.091119,0.763112,True


In [11]:
# пост-хок тест по Стрессоустойчивости между независимыми группами
pg.pairwise_tests(data=df, dv='Scores', between='Group', padjust='bonf') #если дисперсии равны
# pg.pairwise_tests(data=df, dv='Scores', between='Group', padjust='bonf', correction=True) #если дисперсии не равны

,Contrast,A,B,Paired,Parametric,T,dof,alternative,p-unc,BF10,hedges
0,Group,Control,Meditation,False,True,-2.289903,178.0,two-sided,0.0232,1.813,-0.339918


In [12]:
# пост-хок тест по Стрессоустойчивости повторные замеры во времени (ОБЕ ГРУППЫ!)
pg.pairwise_tests(data=df, dv='Scores', within='Time', subject='Subject', padjust='bonf')

,Contrast,A,B,Paired,Parametric,T,dof,alternative,p-unc,p-corr,p-adjust,BF10,hedges
0,Time,August,January,True,True,-1.740370,59.0,two-sided,0.087008,0.261023,bonf,0.582,-0.327583
1,Time,August,June,True,True,-2.743238,59.0,two-sided,0.008045,0.024134,bonf,4.232,-0.482547
2,Time,January,June,True,True,-1.023620,59.0,two-sided,0.310194,0.930581,bonf,0.232,-0.169520


In [13]:
for g in df['Group'].unique():
    res = pg.pairwise_tests(data=df[df['Group']==g], dv='Scores', within='Time', subject='Subject',padjust='bonf')
    print(f"\nPost-hoc внутри группы {g}:\n", res)


Post-hoc внутри группы Control:
   Contrast        A        B  Paired  Parametric         T   dof alternative  \
0     Time   August  January    True        True -0.387729  29.0   two-sided   
1     Time   August     June    True        True -0.297291  29.0   two-sided   
2     Time  January     June    True        True  0.050685  29.0   two-sided   

      p-unc  p-corr p-adjust   BF10    hedges  
0  0.701048     1.0     bonf  0.208 -0.097511  
1  0.768363     1.0     bonf  0.203 -0.074366  
2  0.959924     1.0     bonf  0.195  0.011400  

Post-hoc внутри группы Meditation:
   Contrast        A        B  Paired  Parametric         T   dof alternative  \
0     Time   August  January    True        True -2.022993  29.0   two-sided   
1     Time   August     June    True        True -4.374791  29.0   two-sided   
2     Time  January     June    True        True -1.438842  29.0   two-sided   

      p-unc    p-corr p-adjust     BF10    hedges  
0  0.052379  0.157138     bonf    1.153 -0.

In [17]:
res_time = pd.DataFrame()

for t in df['Time'].unique():
    temp = pg.pairwise_tests(data=df[df['Time']==t], dv='Scores', between='Group', padjust='bonf')
    temp = temp.assign(month = t)
    res_time = pd.concat([temp, res_time])
    del temp
    # print(f"\nPost-hoc для time={t}:\n", res)

res_time

,Contrast,A,B,Paired,Parametric,T,dof,alternative,p-unc,BF10,hedges,month
0,Group,Control,Meditation,False,True,-2.744291,58.0,two-sided,0.008058,5.593,-0.699371,June
0,Group,Control,Meditation,False,True,-1.433725,58.0,two-sided,0.157020,0.619,-0.365379,January
0,Group,Control,Meditation,False,True,0.316022,58.0,two-sided,0.753120,0.274,0.080537,August


In [33]:
# res_time.loc[:, ['month'] + [c for c in list(res_time.columns) if c != 'month']]